# Extract the handbook with Mistral Document AI

This notebook uses the native Mistral OCR client and stores page-level Markdown. The 403-page PDF is split locally into batches below the API's 30-page limit, and every returned page is checkpointed so interrupted runs can resume.

## 1. Imports and project setup

In [2]:
import base64
import json
import random
import sys
import time
from io import BytesIO
from pathlib import Path

from IPython.display import Markdown, display
from mistralai.client import Mistral
from pypdf import PdfReader, PdfWriter
from truststore import inject_into_ssl

inject_into_ssl()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "indexer"
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError("Run this notebook from indexer/notebooks or the repository root.")
sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import IndexerSettings  # noqa: E402

## 2. Configuration

The existing indexer API key and base URL are reused. The settings object is deliberately not printed because it contains credentials.

In [3]:
settings = IndexerSettings()
if not settings.openai_api_key:
    raise ValueError("OPENAI_API_KEY is not configured for the indexer.")
if not settings.openai_api_base:
    raise ValueError("OPENAI_API_BASE is not configured for the indexer.")

DOCUMENT_MODEL = "mistral-document-ai-2512"
PDF_PATH = PROJECT_ROOT / "notebooks" / "Benutzerhilfe Fabasoft eGoc-Suite BAY Okt2025 (1).pdf"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "handbuch_extraction"
PAGES_DIR = OUTPUT_DIR / "pages"
BATCH_SIZE = 20  # The service supports at most 30 PDF pages per request.
REQUEST_TIMEOUT_MS = 180_000
MAX_ATTEMPTS = 5

if not PDF_PATH.is_file():
    raise FileNotFoundError(PDF_PATH)
OUTPUT_DIR.mkdir(exist_ok=True)
PAGES_DIR.mkdir(exist_ok=True)

reader = PdfReader(PDF_PATH)
TOTAL_PAGES = len(reader.pages)
client = Mistral(
    api_key=settings.openai_api_key,
    server_url=settings.openai_api_base.rstrip("/"),
    timeout_ms=REQUEST_TIMEOUT_MS,
)

print(f"Input: {PDF_PATH.name} ({PDF_PATH.stat().st_size / 1024**2:.1f} MiB, {TOTAL_PAGES} pages)")
print(f"OCR endpoint: {settings.openai_api_base.rstrip('/')} /v1/ocr")
print(f"Existing checkpoints: {len(list(PAGES_DIR.glob('page-*.json')))}")

Input: Benutzerhilfe Fabasoft eGoc-Suite BAY Okt2025 (1).pdf (9.7 MiB, 403 pages)
OCR endpoint: https://ki-proxy-test.muenchen.de /v1/ocr
Existing checkpoints: 0


## 3. Batch and checkpoint helpers

In [4]:
def pdf_pages_as_data_url(pdf_reader: PdfReader, page_indexes: list[int]) -> str:
    writer = PdfWriter()
    for page_index in page_indexes:
        writer.add_page(pdf_reader.pages[page_index])
    buffer = BytesIO()
    writer.write(buffer)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:application/pdf;base64,{encoded}"


def checkpoint_path(page_number: int) -> Path:
    return PAGES_DIR / f"page-{page_number:04d}.json"


def page_value(page, name: str, default=None):
    value = getattr(page, name, default)
    return default if value is None else value


def json_value(value):
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json")
    return value


def process_batch(page_indexes: list[int]):
    document = {
        "type": "document_url",
        "document_url": pdf_pages_as_data_url(reader, page_indexes),
        "document_name": f"handbuch-pages-{page_indexes[0] + 1}-{page_indexes[-1] + 1}.pdf",
    }
    return client.ocr.process(
        model=DOCUMENT_MODEL,
        document=document,
        include_image_base64=False,
        include_blocks=False,
        timeout_ms=REQUEST_TIMEOUT_MS,
    )


def checkpoint_response(response, page_indexes: list[int]) -> None:
    if len(response.pages) != len(page_indexes):
        raise RuntimeError(
            f"OCR returned {len(response.pages)} pages for a {len(page_indexes)}-page request."
        )
    for source_index, page in zip(page_indexes, response.pages, strict=True):
        markdown = page.markdown.strip()
        if not markdown:
            raise RuntimeError(f"OCR returned empty Markdown for PDF page {source_index + 1}.")
        record = {
            "page_number": source_index + 1,
            "markdown": markdown,
            "header": page_value(page, "header"),
            "footer": page_value(page, "footer"),
            "tables": [json_value(table) for table in page_value(page, "tables", [])],
            "hyperlinks": [json_value(link) for link in page_value(page, "hyperlinks", [])],
            "dimensions": json_value(page_value(page, "dimensions")),
            "source": PDF_PATH.name,
            "model": response.model,
        }
        checkpoint_path(source_index + 1).write_text(
            json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8"
        )

## 4. Probe the native OCR endpoint

Run one page first. A successful response is checkpointed and previewed before the full extraction starts.

In [5]:
print(f"Probing native OCR model: {DOCUMENT_MODEL}")
probe_response = process_batch([0])
checkpoint_response(probe_response, [0])
print(f"Page 1 extracted with {probe_response.model}.")
display(Markdown(probe_response.pages[0].markdown[:4000]))

Probing native OCR model: mistral-document-ai-2512
Page 1 extracted with mistral-document-ai-2512.


Fabasoft

![img-0.jpeg](img-0.jpeg)

# Fabasoft eGov-Suite 2025

Benutzerhilfe

Gültig ab 19. Oktober 2025

## 5. Extract and checkpoint all pages

Only pages without checkpoints are submitted. Each batch is retried with exponential backoff and its pages are persisted immediately after a successful response. Re-run this cell to resume.

In [6]:
pending_indexes = [index for index in range(TOTAL_PAGES) if not checkpoint_path(index + 1).is_file()]
failed_batches = []

for offset in range(0, len(pending_indexes), BATCH_SIZE):
    page_indexes = pending_indexes[offset : offset + BATCH_SIZE]
    page_label = f"{page_indexes[0] + 1}-{page_indexes[-1] + 1}"
    last_error = None
    print(f"Processing PDF pages {page_label} ({len(page_indexes)} pages)")

    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            response = process_batch(page_indexes)
            checkpoint_response(response, page_indexes)
            break
        except Exception as exc:
            last_error = exc
            if attempt == MAX_ATTEMPTS:
                failed_batches.append({"pages": page_label, "error": f"{type(exc).__name__}: {exc}"})
                print(f"Batch {page_label} failed and was left for the next run.")
                break
            delay = min(120, 5 * 2 ** (attempt - 1)) + random.uniform(0, 2)
            print(f"Batch {page_label}: attempt {attempt} failed; retrying in {delay:.1f}s")
            time.sleep(delay)

(OUTPUT_DIR / "failures.json").write_text(
    json.dumps(failed_batches, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Finished this run with {len(failed_batches)} failed batch(es).")

Processing PDF pages 2-21 (20 pages)
Processing PDF pages 22-41 (20 pages)
Processing PDF pages 42-61 (20 pages)
Processing PDF pages 62-81 (20 pages)
Processing PDF pages 82-101 (20 pages)
Processing PDF pages 102-121 (20 pages)
Processing PDF pages 122-141 (20 pages)
Processing PDF pages 142-161 (20 pages)
Processing PDF pages 162-181 (20 pages)
Processing PDF pages 182-201 (20 pages)
Processing PDF pages 202-221 (20 pages)
Processing PDF pages 222-241 (20 pages)
Processing PDF pages 242-261 (20 pages)
Processing PDF pages 262-281 (20 pages)
Processing PDF pages 282-301 (20 pages)
Processing PDF pages 302-321 (20 pages)
Processing PDF pages 322-341 (20 pages)
Processing PDF pages 342-361 (20 pages)
Processing PDF pages 362-381 (20 pages)
Processing PDF pages 382-401 (20 pages)
Processing PDF pages 402-403 (2 pages)
Finished this run with 0 failed batch(es).


## 6. Validate and combine the page results

This cell refuses to create a final document if pages are missing, duplicated, or empty.

In [7]:
records = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(PAGES_DIR.glob("page-*.json"))]
page_numbers = [record["page_number"] for record in records]
missing_pages = sorted(set(range(1, TOTAL_PAGES + 1)) - set(page_numbers))
duplicate_pages = sorted({number for number in page_numbers if page_numbers.count(number) > 1})
empty_pages = [record["page_number"] for record in records if not record["markdown"].strip()]
short_pages = [record["page_number"] for record in records if len(record["markdown"].strip()) < 40]

print(f"Extracted pages: {len(records)}/{TOTAL_PAGES}")
print(f"Missing: {missing_pages or 'none'}")
print(f"Duplicates: {duplicate_pages or 'none'}")
print(f"Empty: {empty_pages or 'none'}")
print(f"Suspiciously short: {short_pages or 'none'}")

if missing_pages or duplicate_pages or empty_pages:
    raise RuntimeError("Extraction is incomplete; re-run the extraction cell before combining results.")

records.sort(key=lambda record: record["page_number"])
combined_markdown = "\n\n".join(
    f"<!-- PDF page {record['page_number']} -->\n\n{record['markdown']}" for record in records
)
json_path = OUTPUT_DIR / "handbuch-pages.json"
markdown_path = OUTPUT_DIR / "handbuch.md"
json_path.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
markdown_path.write_text(combined_markdown, encoding="utf-8")
print(f"Wrote {json_path}")
print(f"Wrote {markdown_path}")

Extracted pages: 403/403
Missing: none
Duplicates: none
Empty: none
Suspiciously short: [146, 183, 276, 277, 278, 279, 381, 383, 384, 387]
Wrote c:\Users\sebastian.berger\projects\opensource-projects\snowman\indexer\notebooks\handbuch_extraction\handbuch-pages.json
Wrote c:\Users\sebastian.berger\projects\opensource-projects\snowman\indexer\notebooks\handbuch_extraction\handbuch.md


## 7. Preview selected pages

In [9]:
for page_number in sorted({1, max(1, TOTAL_PAGES // 2), 5}):
    record = records[page_number - 1]
    display(Markdown(f"## PDF page {page_number}\n\n{record['markdown'][:4000]}"))

## PDF page 1

Fabasoft

![img-0.jpeg](img-0.jpeg)

# Fabasoft eGov-Suite 2025

Benutzerhilfe

Gültig ab 19. Oktober 2025

## PDF page 5

Benutzerhilfe
5

# 2 Supportprozesse
47

## 2.1 Allgemeine Anwendungsfälle
47

### 2.1.1 Umgang mit Objekten
47

### 2.1.2 Hinweis: Die Dublettenprüfung kommt auch beim Erzeugen von Schlagwerten zum Einsatz. Für weitere Informationen zu Schlagwerten siehe Kapitel 2.6.8 „Synchronisierungs-Ausnahmen“
49

### 2.1.3 Hervorheben von Objekten
56

### 2.1.4 Versenden von Objekten
56

### 2.1.5 Kontextmenü
58

### 2.1.6 Spalteneinstellungen
58

### 2.1.7 Reihenfolge einer Objektliste verändern
63

### 2.1.8 Objektlisten
64

### 2.1.9 Umgang mit Aggregaten
69

## 2.2 Adressatenverwaltung
71

### 2.2.1 Globales Adressverzeichnis
71

### 2.2.2 Allgemeine Informationen zur Adressatenverwaltung
72

### 2.2.3 Rechtskonzept Kontaktraum
72

### 2.2.4 Einen Kontaktraum erzeugen
73

### 2.2.5 Ordner im Kontaktraum
73

### 2.2.6 Einen Adressaten erzeugen
74

### 2.2.7 Einen Verteiler erzeugen
74

### 2.2.8 Eine Organisation erzeugen
74

### 2.2.9 Einen Adressaten bearbeiten
74

### 2.2.10 Adressaten einem anderen Kontaktraum zuweisen
75

### 2.2.11 Schaltfläche Auswahl Adresstyp
75

## 2.3 Umgang mit Dokumenten
78

### 2.3.1 Ein Dokument erstellen
78

### 2.3.2 Die Schriftstücke eines Dokuments lesen oder bearbeiten
78

## 2.4 Öffnen des zugehörigen Objekts
79

## 2.5 IntelliHelp
80

## 2.6 Synchronisierungsfunktionalität „Folio Ordner“
81

### 2.6.1 Aktivierung der Synchronisierung
81

### 2.6.2 Synchronisierung erstmalig verwenden
82

### 2.6.3 Ordner und Dokumente für die Synchronisierung auswählen
83

### 2.6.4 Symbole für die Visualisierung des Status
84

### 2.6.5 Auflösen von Konflikten
85

### 2.6.6 Erweiterte Möglichkeiten
85

## PDF page 201

Benutzerhilfe

# 3.4.2 Geschäftszeichenbildung einer Erledigung

Das Geschäftszeichen einer Erledigung bildet sich aus den folgenden Attributen:

- Der zuständigen Organisationseinheit
- Dem Geschäftszeichen des Vorgangs
- Einer fortlaufenden Nummer

Das Geschäftszeichen kann über folgende Möglichkeiten geändert werden:

- Das Feld Zuständige Organisationseinheit wird geändert,
- die Erledigung wird einem anderen Vorgang oder
- der Vorgang zu einer anderen Akte umgeschrieben.

## Ändern des Geschäftszeichens mittels Änderung der zuständigen Organisationseinheit

1. Öffnen Sie die Eigenschaften der Erledigung im Bearbeitungsmodus.
2. Wählen Sie in dem Feld Zuständige Organisationseinheit die gewünschte Organisationseinheit.
**Hinweis:** Sie können in diesem Feld auch eine Suche durchführen.
3. Verwenden Sie die Schaltfläche „Weiter“, um Ihre Änderungen zu speichern.

## Änderung des Geschäftszeichens mittels Umschreibung

Um das Geschäftszeichen einer Erledigung mittels einer Umschreibung zu ändern, bringen Sie die Unterschrift „Umschreibung“ auf das Eingangsdokument oder den Vorgang, wie im Kapitel 3.3.4 „Einen Vorgang umschreiben“ bzw. Kapitel 2.11 „Unterschreiben“ beschrieben, an.

# 3.4.3 Einen Empfänger einer Erledigung definieren

Im Feld Adressaten in den Metadaten einer Erledigung werden die Empfänger angegeben, an die die Reinschrift der Erledigung versandt werden soll. Für weitere Informationen bezüglich der Bearbeitung eines Aggregats wechseln Sie ins Kapitel 2.1.9 „Umgang mit Aggregaten“.

Um einen neuen Empfänger einer Erledigung zu definieren gehen Sie wie folgt vor:

1. Lokalisieren Sie die Erledigung, welcher Sie Empfänger hinzufügen möchten.
2. Öffnen Sie die Eigenschaften der Erledigung über die Kontextmenüaktion „Eigenschaften“.
3. Wechseln Sie auf die Registerkarte „Adressaten“.
4. Klicken Sie auf „Bearbeiten“, sofern die Eigenschaften lesend geöffnet wurden.
5. Fügen Sie über die Schaltfläche „Eintrag hinzufügen“ neue Empfänger im Aggregat „Adressaten“ hinzu. (siehe auch Kapitel 2.1.9 „Umgang mit Aggregaten)

201